### Developing an alpha in worldquant brain

Developing an alpha in *worldquant brain* involves a structured quantitative research pipeline that can be sumarised into the following steps : 

1. *Formulate an economic hypothesis*

- In quantitative finance, an *alpha hypothesis* is a testable, economically grounded statement about market behavior or asset pricing inefficiencies.It can be simplified to as an idea which aims to capture market inefficiencies.It serves as the blueprint of creating your alpha expression.

- An example of an hypothesis is *PRICE REVERSION*.This is where you *"Buy stocks whose prices fell recently, expecting them to revert to their average historical price; short stocks that experienced a rise in price last week, expecting them to fall back to normal."*

2. *Select Data Fields and Universe*

- Choose the appropriate Region (e.g., USA) and Universe (e.g., TOP3000, TOP1000, TOP500) based on liquidity needs. Identify the specific Data Fields required—such as `price`/`volume` (`close`, `returns`, `vwap`), fundamentals, analyst estimates, or sentiment datasets—ensuring they align with your hypothesis timeframe.

3. *Translate into a Fast Expression*

- Write the mathematical formula in BRAIN’s Fast Expression language:

- Capture the core effect using operators like `rank()`, `ts_delta()`, `ts_zscore()`, or `ts_decay_linear()` etc.

4. *Configure Simulation Settings*

- Set your simulation parameters carefully before running backtests:

- Decay: Smooth daily weight changes (e.g., `Decay = 4` to `10`) to reduce transaction costs and turnover.

- Neutralization: Strip out unwanted broad market or sector exposure using Subindustry, Industry, Sector, or Market neutralization.

- Truncation: Cap maximum single-stock weight (typically `0.01` to `0.05`) to prevent concentration risk.

- Delay: Use `Delay 1` (previous day's data) for standard end-of-day signals, or `Delay 0` only when strictly justified.

5. *Backtest and Evaluate Metrics*

- Run the simulation on the In-Sample (IS) period and verify that your alpha clears BRAIN's core submission thresholds:

- *Sharpe Ratio*: Generally `> 1.25` (for `Delay 1`) to ensure strong risk-adjusted returns.

- *Fitness: Generally* `> 1.0` (balances *Sharpe, returns, and turnover*).

- *Turnover*: Typically between `1%` and `70%` so returns aren't eroded by trading costs.

- *Weight & Sub-universe Checks*: Ensure weights are well-distributed across stocks and the signal holds up in smaller, more liquid sub-universes (e.g., `TOP500`).

6. *Refine Without Overfitting*

- If metrics fall short, adjust the economic logic rather than blindly sweeping lookback windows (e.g., testing every integer from 5 to 60). Verify that nearby parameter values (like 20-day vs. 30-day lookbacks) produce stable, consistent PnL curves and check Self-Correlation (`< 0.7`) against your existing submitted alphas.

7. *Submit to Production*

- Once all IS checks pass, click Submit. BRAIN automatically tests your alpha against unseen Out-of-Sample data and checks production correlation against the global pool before adding it to your consultant portfolio.

### Example

Lets start with the *HYPOTHESIS*

- The main idea is that "Stocks that moved a lot recently tend to snap back more, so bet on the reversal, and bet bigger on the ones that are most volatile."

- In practice: if a stock has fallen recently, buy it. If it has risen, short it. The more volatile the stock, the larger the position.

Some of the terminologies you need to know :
1. Taking a `long position` means buying a stock first and selling it later. You profit if the price goes up. Example: buy at $100, sell at $120, and you make $20 per share. If it drops to $80, you lose $20.

2. `Shorting` means selling first and buying back later. You borrow shares from your broker, sell them at today’s price, and hope to buy them back cheaper. Example: borrow and sell at $100, buy back at $80, and you keep the $20 difference. If it rises to $120, you lose $20.

3. `volatility` is how much a stock’s price bounces around over time. It’s a measure of how big and how frequent the price swings are, not which direction they go.

- A stock that moves 0.5% a day and rarely surprises you has low volatility. A stock that jumps 5% up one day and drops 4% the next has high volatility.

Moving on to step 2 where we *Select Data Fields and Universe* we will use the following:

- TOP 3000 universe 
- returns 

Log into worldquant and go to the simulation tab.

1. start by simulating `-returns` 
    - *what is returns?*
    - In this context, returns is the percentage change in a stock’s price from one day to the next.

    - Formula: `(today's close − yesterday's close) / yesterday's close`

    - Example: if a stock closes at $100 yesterday and $103 today, the return is 3%, or 0.03.

    - Returns are used instead of raw prices because they’re comparable across stocks. A $5 stock rising $0.15 and a $500 stock rising $15 both had a 3% return.

    - The negative sign inverts the results of `returns` 
    - here is a simple task : *try simulating `returns` and compare its graph withe the one for `-returns`*

**Simulation Settings**

Language: Fast Expression

Instrument Type: Equity

Region: USA

Delay: 1

Universe: TOP3000

Neutralization: Subindustry

Decay: 4

Truncation: 0.08

Pasteurization: On

Unit Handling: Verify

NaN Handling: Off

Test Period: 1 Years, 0 Months

The results after simulating `-returns` is as follows :


![-returns simulation](./images/returns.png)

### After simulating 
You will notice that the alpha is not submitable since it has not met some of the requirements

![- returns metrics](./images/returnsinfo.png)

*`So lets work on improving the alpha`* 


### Improving the alpha by introducing a new operator `ts_decay_linear(x,d)`

This takes a weighted average of the last 30 days, giving recent days more weight, with weights falling in a straight line. A single bad day counts for less than a steady trend.

- Here `x` is a time series (an ordered sequence of values across time for each asset/instrument—typically daily bars in quant backtesting). It can be:
- Raw market or fundamental data: e.g., `close`, `volume`, `returns`, or a fundamental metric.
- Then `d` is the lookback period in days.

1. WHY WE USE `ts_decay_linear(x,d)`.
- Smooths noisy signals while staying responsive: Because recent values carry more weight than older ones, your alpha reacts faster to fresh information than a plain rolling average, while still damping single-day spikes.
- Lowers turnover: A smoother signal produces smaller day-to-day weight swings in your simulated portfolio, which helps reduce turnover (and often improves Sharpe/Fitness when a raw signal flips around too much).   
- Models gradual fading effects: Useful when a signal or event's relevance naturally tapers off over a set number of trading days rather than cutting off abruptly.

2. Implementation using `ts_decay_linear(x,d)`.

- Taking our `x` as `-returns` with lookback period of `30` days therefore `d = 30`
- Therefore full expresion as `ts_decay_linear(-returns,30)` ;
- we get an average alpha signal with better IS score 
- Try simulating it as below:

![ts_decay_linear(-returns,30)_graph](./images/ts_decay_graph.png)

As you can see the alphas status improved from `needs improvement` to `average`

### Now lets look at IS TESTING STATUS

![ts_decay_linear_IS_status](./images/ts_decay_is.png)


WE NEED TO IMPROVE SHARPE OF THE ALPHA

To improve the alpha a bit i introduce `rank(ts_std_dev(close, 30))`

What it does, step by step

- Step 1: ts_std_dev(close, 30)
For each stock, take its last 30 closing prices and measure how spread out they are (standard deviation). A stock that stays near $100 has a small number. A stock that swings between $60 and $140 has a large number.

- Step 2: rank(...)
Each day, look at every stock’s number from Step 1, sort them from smallest to largest, and convert each into a position from 0 to 1.

- Calmest stock in the universe → near 0
Jumpiest stock → near 1
Middle of the pack → about 0.5

- The output is a relative ranking, not a raw measure. A stock with a standard deviation of 15 might rank 0.9 if the rest of the market is calmer, and 0.4 on a day when the market is more volatile.

Why the ranking helps

1. It makes the numbers comparable. Raw standard deviations depend on the price level. A $500 stock will almost always have a larger number than a $20 stock, even if both are equally jumpy. Ranking removes the scale.

2. It keeps the scale stable over time. During a calm market, all standard deviations shrink. During a volatile market, they all grow. Ranking absorbs that shift, so the weights stay in the same range from one day to the next.

3. It limits extreme bets. A single stock with an unusually large standard deviation can’t dominate. The highest rank is capped at 1.

4. It makes the result a relative choice. The question becomes “which stocks are jumpier than their peers?” rather than “how jumpy is this stock in absolute terms?”

The hypothesis behind it

The idea is that jumpy stocks overreact more, so they revert more strongly.

Here’s the chain of reasoning:

When a stock is jumpy, its price is moving a lot, often because traders are emotional, news is heavy, or there’s a lot of selling or buying pressure.
Some of those moves are overreactions. The price goes further than the company’s real value justifies.
When the pressure fades, the price tends to snap back. The bigger the overreaction, the bigger the snap-back.
So a jumpy stock that has just fallen should bounce harder than a calm stock that has just fallen.

The volatility rank is a way of saying: “give more weight to the stocks where we expect the biggest snap-backs.”

*THUS THE FINAL ALPHA EXPRESSION*

- `ts_decay_linear(-returns,30) * (1 * rank(ts_std_dev(close,30)))`

![final-alpha-graph](./images/final_alpha_graph.png)

The IS testing status

![final-alpha-IS](./images/final_alpha_is1.png)


NOTE : If you’ve followed the examples in the Starter pack, chances are you’ve ended up with the first 2 graphs. What both graphs have in common is that they have multiple periods of significant losses, producing a graph with high fluctuations. This means that your simulated portfolio could lose a large percentage of its value in one day, and that wouldn’t be ideal. Rather, a good Alpha should produce a steadily rising PnL chart (3rd graph) with few fluctuations and no major drawdown.

![good-graphs-image](./images/good-graph.png)